In [7]:
!pip install transformers torch peft sentencepiece -q

In [8]:
!pip install torchao --upgrade -q

In [6]:
from google.colab import files
import zipfile, os

# 1. Upload le zip
uploaded = files.upload()  # sélectionne model_finetuned.zip

# 2. Extraire
with zipfile.ZipFile("model_finetuned.zip", "r") as z:
    z.extractall(".")

# 3. Vérifier
print(os.listdir("model_finetuned"))  # doit afficher adapter_config.json etc.

Saving model_finetuned.zip to model_finetuned (1).zip
['README.md', 'tokenizer.json', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json']


In [9]:
import json, os, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

# Charger le modèle fine-tuné de ta collègue
MODEL_NAME   = "google/pegasus-xsum"
MODEL_DIR    = "model_finetuned"

print("🔄 Chargement du modèle PEGASUS + LoRA...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model      = PeftModel.from_pretrained(base_model, MODEL_DIR)
tokenizer  = AutoTokenizer.from_pretrained(MODEL_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device)
model.eval()

print(f"✅ Modèle chargé sur {device.upper()}")

🔄 Chargement du modèle PEGASUS + LoRA...


Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Modèle chargé sur CUDA


In [10]:
from google.colab import files
uploaded = files.upload()  # sélectionne donnees_propres.json

Saving donnees_propres.json to donnees_propres.json


In [11]:
import json, os

# Détecter le bon chemin
if os.path.exists("donnees_propres.json"):
    chemin = "donnees_propres.json"
elif os.path.exists("notebooks/donnees_propres.json"):
    chemin = "notebooks/donnees_propres.json"
else:
    raise FileNotFoundError("❌ donnees_propres.json introuvable !")

with open(chemin, "r", encoding="utf-8") as f:
    donnees = json.load(f)

print(f"✅ {len(donnees)} livres chargés")
print(f"   Exemple 1 : {donnees[0]['nb_chunks']} chunks")

✅ 2000 livres chargés
   Exemple 1 : 14 chunks


In [12]:
def resumer_chunk(chunk, max_input=512, max_output=128):
    inputs = tokenizer(
        chunk,
        max_length=max_input,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_output,
            num_beams=2,
            early_stopping=True,
            no_repeat_ngram_size=3,
            forced_bos_token_id=None   # ← supprime le conflit max_length
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("✅ Fonction resumer_chunk définie !")

✅ Fonction resumer_chunk définie !


In [13]:
def pipeline_mapreduce(exemple):
    """
    MAP    : résumer chaque chunk individuellement
    REDUCE : résumer tous les résumés en un résumé final
    """
    chunks = exemple["chunks"]
    print(f"   📚 {len(chunks)} chunks à traiter...")

    # ── MAP : résumer chaque chunk ──
    resumes_intermediaires = []
    for i, chunk in enumerate(chunks):
        print(f"   🔄 Chunk {i+1}/{len(chunks)}...", end=" ")
        resume = resumer_chunk(chunk)
        resumes_intermediaires.append(resume)
        print(f"✅ ({len(resume.split())} mots)")

    # ── REDUCE : combiner les résumés intermédiaires ──
    texte_combine = " ".join(resumes_intermediaires)
    print(f"   🔄 Reduce : résumé final...")
    resume_final = resumer_chunk(texte_combine, max_input=512, max_output=150)

    return {
        "resumes_chunks": resumes_intermediaires,
        "resume_final":   resume_final
    }

print("✅ Fonction pipeline_mapreduce définie !")

✅ Fonction pipeline_mapreduce définie !


In [14]:
print("=" * 55)
print("TEST PIPELINE MAP-REDUCE sur 3 livres")
print("=" * 55)

resultats = []

for i in range(3):
    print(f"\n📖 Livre {i+1} :")
    res = pipeline_mapreduce(donnees[i])

    print(f"\n   ✅ Résumé final ({len(res['resume_final'].split())} mots) :")
    print(f"   {res['resume_final']}")
    print(f"\n   📌 Référence :")
    print(f"   {donnees[i]['resume'][:200]}...")

    resultats.append({
        "index":        i,
        "resume_final": res["resume_final"],
        "reference":    donnees[i]["resume"]
    })

print("\n✅ Pipeline Map-Reduce terminé !")

TEST PIPELINE MAP-REDUCE sur 3 livres

📖 Livre 1 :
   📚 14 chunks à traiter...
   🔄 Chunk 1/14... ✅ (114 mots)
   🔄 Chunk 2/14... ✅ (106 mots)
   🔄 Chunk 3/14... ✅ (83 mots)
   🔄 Chunk 4/14... ✅ (109 mots)
   🔄 Chunk 5/14... ✅ (107 mots)
   🔄 Chunk 6/14... ✅ (113 mots)
   🔄 Chunk 7/14... ✅ (107 mots)
   🔄 Chunk 8/14... ✅ (112 mots)
   🔄 Chunk 9/14... ✅ (118 mots)
   🔄 Chunk 10/14... ✅ (115 mots)
   🔄 Chunk 11/14... ✅ (112 mots)
   🔄 Chunk 12/14... ✅ (109 mots)
   🔄 Chunk 13/14... ✅ (114 mots)
   🔄 Chunk 14/14... ✅ (109 mots)
   🔄 Reduce : résumé final...

   ✅ Résumé final (68 mots) :
   On the morning of the first day of battle, the British soldiers are told that the French have been planning to attack the forts of Fort Edward and Fort Ticonderoga, and that they have been ordered to prepare for a major battle. The soldiers of the Army of the American Indians, who are also known as the Indians, have been preparing for the battle for a long time.

   📌 Référence :
   Before any characte

In [15]:
import json

# Sauvegarder les résultats
with open("resultats_mapreduce.json", "w", encoding="utf-8") as f:
    json.dump(resultats, f, indent=2, ensure_ascii=False)

print("✅ resultats_mapreduce.json sauvegardé !")
print(f"   {len(resultats)} livres résumés")

✅ resultats_mapreduce.json sauvegardé !
   3 livres résumés


In [16]:
from google.colab import files
files.download("resultats_mapreduce.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
import shutil
from google.colab import files

# Zipper le dossier model_finetuned
shutil.make_archive("model_finetuned", "zip", ".", "model_finetuned")
print("✅ model_finetuned.zip créé !")

# Télécharger aussi le rapport
files.download("model_finetuned.zip")
files.download("rapport_finetuning.json")

✅ model_finetuned.zip créé !


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FileNotFoundError: Cannot find file: rapport_finetuning.json